In [1]:
# ==============================================================================
# SHAURYA'S NOTEBOOK: MODEL 2 - PRODUCTION SHORTFALL PREDICTOR (XGBoost)
# ==============================================================================

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# 1. Load Data
df = pd.read_csv("production_shortfall_data.csv")
print(f"Loaded {len(df)} monthly production records across MOIL mines.")

# 2. Encode categorical 'mine_type' (Opencast=1, Underground=0)
df['is_opencast'] = (df['mine_type'] == 'Opencast').astype(int)

# 3. Features & Target
feature_cols = [
    'monthly_target_tonnes',
    'rainfall_mm',
    'avg_temperature_c',
    'equipment_uptime_pct',
    'breakdown_hours',
    'blasting_delays_count',
    'avg_feed_grade_pct',
    'is_opencast',
    'month'
]

X = df[feature_cols]
y = df['shortfall_pct']  # Predicting % shortfall

# 4. Train/Test Split (Temporal/Random holdout)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# 5. Train XGBoost Regressor
print("\n--- Training XGBoost Production Shortfall Model ---")
reg = xgb.XGBRegressor(
    n_estimators=120,
    max_depth=4,
    learning_rate=0.06,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42
)
reg.fit(X_train, y_train)

# 6. Evaluate Model
y_pred = reg.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f}%")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}%")
print(f"R² Score: {r2:.4f}")

# 7. Feature Importance (Root causes of Shortfall)
print("\nTop Root Causes Driving Production Shortfalls:")
imp = pd.Series(reg.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(imp)

# 8. Save Trained Model
joblib.dump(reg, "model2_shortfall.pkl")
print("\nSaved 'model2_shortfall.pkl' successfully!")

Loaded 360 monthly production records across MOIL mines.

--- Training XGBoost Production Shortfall Model ---
Mean Absolute Error (MAE): 3.27%
Root Mean Squared Error (RMSE): 4.54%
R² Score: 0.7895

Top Root Causes Driving Production Shortfalls:
breakdown_hours          0.295002
is_opencast              0.170770
equipment_uptime_pct     0.156590
rainfall_mm              0.149307
month                    0.091157
avg_feed_grade_pct       0.048280
blasting_delays_count    0.039902
avg_temperature_c        0.030425
monthly_target_tonnes    0.018567
dtype: float32

Saved 'model2_shortfall.pkl' successfully!
